# Screen & Natural Images Classification Baselines
In this notebook, we run kNN and Random Forest on Enrico screen images and Caltech101 natural images as baselines.

In [1]:
import sys
import os

import torch
from torchvision import datasets
from torchvision.transforms import v2
from sklearn.model_selection import train_test_split
import numpy as np
from screen_images import CustomEnricoDataset, get_allowed_classes, make_screen_base_transform
from classification_baseline import extract_features, train_evaluate_knn, train_evaluate_random_forest

## Data Loading and Preprocessing
We use a smaller image size (60x40) for classical ML algorithms to keep the feature dimension manageable for both Enrico and Caltech101 datasets.

In [2]:
# Data configuration
BATCH_SIZE = 128
ENRICO_CLASSES = len(get_allowed_classes())

# We resize images for basic machine learning models so that flattened feature vectors
# are not overly huge (e.g., 60x40 means 60*40*3 = 7200 features)
base_preprocess = make_screen_base_transform(resize=(60, 40))

# ----------------------------------------------------
# 1. Load Enrico Dataset
# ----------------------------------------------------
enrico_root = "/kaggle/input/datasets/nazariyyuchnovskiy/enricoscreenshotsandwireframes"
if not os.path.exists(enrico_root):
    enrico_root = "./data/enricoscreenshotsandwireframes"

try:
    print(f"Loading Enrico data from: {enrico_root}")
    enrico_train_dataset, enrico_val_dataset, enrico_test_dataset = CustomEnricoDataset.create_splits(
        root=enrico_root,
        val_size=0.1,
        test_size=0.1,
        use_wireframes=False,
        train_transform=base_preprocess,
        eval_transform=base_preprocess
    )

    enrico_train_loader = torch.utils.data.DataLoader(enrico_train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    enrico_test_loader = torch.utils.data.DataLoader(enrico_test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print("Enrico dataset splits and loaders created successfully!")
except Exception as e:
    print(f"Error loading Enrico dataset: {e}")

# ----------------------------------------------------
# 2. Load Caltech101 Dataset
# ----------------------------------------------------
caltech_transform = v2.Compose([
    v2.Resize(size=(60, 40)),
    v2.RGB(),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
])

caltech_root = "/kaggle/input/datasets/nazariyyuchnovskiy/caltech101"
download_caltech = False
if not os.path.exists(caltech_root):
    caltech_root = "./data"
    download_caltech = True

try:
    print(f"Loading Caltech101 data from: {caltech_root}")
    full_caltech = datasets.Caltech101(root=caltech_root, transform=caltech_transform, download=download_caltech)
    targets = np.array(full_caltech.y)

    train_idx, test_idx = train_test_split(
        np.arange(len(full_caltech)),
        test_size=0.1,
        stratify=targets,
        random_state=42
    )

    caltech_train_dataset = torch.utils.data.Subset(full_caltech, train_idx)
    caltech_test_dataset = torch.utils.data.Subset(full_caltech, test_idx)

    caltech_train_loader = torch.utils.data.DataLoader(caltech_train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    caltech_test_loader = torch.utils.data.DataLoader(caltech_test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    print("Caltech101 dataset splits and loaders created successfully!")
except Exception as e:
    print(f"Error loading Caltech101 dataset: {e}")


Loading Enrico data from: /kaggle/input/datasets/nazariyyuchnovskiy/enricoscreenshotsandwireframes
Enrico dataset splits and loaders created successfully!
Loading Caltech101 data from: ./data


100%|██████████| 137M/137M [00:02<00:00, 59.2MB/s]


Caltech101 dataset splits and loaders created successfully!


## Feature Extraction
Flatten the multidimensional tensors into 1D vectors per image.

In [3]:
# Extract features (flatten images into 1D vectors for scikit-learn models)
print("=== Extracting features for Enrico dataset ===")
print("Extracting features for training data...")
X_train_enrico, y_train_enrico = extract_features(enrico_train_loader, flatten=True)
print("Extracting features for testing data...")
X_test_enrico, y_test_enrico = extract_features(enrico_test_loader, flatten=True)
print(f"Enrico Train data shape: X={X_train_enrico.shape}, y={y_train_enrico.shape}")
print(f"Enrico Test data shape: X={X_test_enrico.shape}, y={y_test_enrico.shape}")

print("\n=== Extracting features for Caltech101 dataset ===")
print("Extracting features for training data...")
X_train_caltech, y_train_caltech = extract_features(caltech_train_loader, flatten=True)
print("Extracting features for testing data...")
X_test_caltech, y_test_caltech = extract_features(caltech_test_loader, flatten=True)
print(f"Caltech101 Train data shape: X={X_train_caltech.shape}, y={y_train_caltech.shape}")
print(f"Caltech101 Test data shape: X={X_test_caltech.shape}, y={y_test_caltech.shape}")


=== Extracting features for Enrico dataset ===
Extracting features for training data...
Extracting features for testing data...
Enrico Train data shape: X=(979, 7200), y=(979,)
Enrico Test data shape: X=(123, 7200), y=(123,)

=== Extracting features for Caltech101 dataset ===
Extracting features for training data...
Extracting features for testing data...
Caltech101 Train data shape: X=(7809, 7200), y=(7809,)
Caltech101 Test data shape: X=(868, 7200), y=(868,)


## k-Nearest Neighbors (kNN) Classifier

In [4]:
# kNN Baseline on Enrico Dataset
print("===========================================")
print("===      kNN Baseline: Enrico Dataset    ===")
print("===========================================")
for i in range(5, 31):
    print(f">>> [Enrico] Training kNN with {i} neighbors:")
    knn_model, knn_preds = train_evaluate_knn(X_train_enrico, y_train_enrico, X_test_enrico, y_test_enrico, n_neighbors=i, n_jobs=-1)

===      kNN Baseline: Enrico Dataset    ===
>>> [Enrico] Training kNN with 5 neighbors:
Training kNN with n_neighbors=5...
Evaluating kNN...
kNN Results -> Accuracy: 0.3008 | Precision: 0.3360 | Recall: 0.3008 | F1: 0.2910

Classification Report (kNN):
              precision    recall  f1-score   support

           0       0.09      0.29      0.14         7
           1       0.14      0.09      0.11        11
           2       0.38      0.20      0.26        15
           3       0.41      0.63      0.50        27
           4       0.14      0.07      0.10        14
           5       0.67      0.50      0.57         8
           6       0.22      0.33      0.27         6
           7       1.00      0.20      0.33         5
           8       0.10      0.11      0.11         9
           9       0.00      0.00      0.00         4
          10       0.45      0.29      0.36        17

    accuracy                           0.30       123
   macro avg       0.33      0.25      0.2

In [5]:
# kNN Baseline on Caltech101 Dataset
print("\n===========================================")
print("===    kNN Baseline: Caltech101 Dataset  ===")
print("===========================================")
for i in range(80, 110):
    print(f">>> [Caltech101] Training kNN with {i} neighbors:")
    knn_model_caltech, knn_preds_caltech = train_evaluate_knn(X_train_caltech, y_train_caltech, X_test_caltech, y_test_caltech, n_neighbors=i, n_jobs=-1)


===    kNN Baseline: Caltech101 Dataset  ===
>>> [Caltech101] Training kNN with 80 neighbors:
Training kNN with n_neighbors=80...
Evaluating kNN...
kNN Results -> Accuracy: 0.3583 | Precision: 0.3254 | Recall: 0.3583 | F1: 0.2863

Classification Report (kNN):
              precision    recall  f1-score   support

           0       0.84      0.61      0.71        44
           1       0.72      1.00      0.84        44
           2       0.10      0.90      0.19        20
           3       0.65      0.84      0.73        80
           4       0.00      0.00      0.00         5
           5       0.29      0.97      0.45        80
           6       0.00      0.00      0.00         4
           7       0.00      0.00      0.00         4
           8       0.00      0.00      0.00         5
           9       0.00      0.00      0.00         5
          10       0.00      0.00      0.00         5
          11       0.00      0.00      0.00         3
          12       0.27      0.23   

## Random Forest Classifier

In [6]:
# Random Forest Baseline on Enrico Dataset
print("===========================================")
print("===  Random Forest Baseline: Enrico     ===")
print("===========================================")
for i in range(85, 115):
    print(f">>> [Enrico] Training RandomForest with {i} estimators:")
    rf_model, rf_preds = train_evaluate_random_forest(X_train_enrico, y_train_enrico, X_test_enrico, y_test_enrico, n_estimators=i, n_jobs=-1, random_state=42)

===  Random Forest Baseline: Enrico     ===
>>> [Enrico] Training RandomForest with 85 estimators:
Training Random Forest with n_estimators=85...
Evaluating Random Forest...
Random Forest Results -> Accuracy: 0.4634 | Precision: 0.4620 | Recall: 0.4634 | F1: 0.4349

Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.57      0.57      0.57         7
           1       0.33      0.18      0.24        11
           2       0.44      0.47      0.45        15
           3       0.49      0.81      0.61        27
           4       0.37      0.50      0.42        14
           5       0.83      0.62      0.71         8
           6       0.33      0.33      0.33         6
           7       1.00      0.40      0.57         5
           8       0.50      0.11      0.18         9
           9       0.00      0.00      0.00         4
          10       0.36      0.29      0.32        17

    accuracy                           0.46 

In [7]:
# Random Forest Baseline on Caltech101 Dataset
print("\n===========================================")
print("=== Random Forest Baseline: Caltech101  ===")
print("===========================================")
for i in range(115, 145):
    print(f">>> [Caltech101] Training RandomForest with {i} estimators:")
    rf_model_caltech, rf_preds_caltech = train_evaluate_random_forest(X_train_caltech, y_train_caltech, X_test_caltech, y_test_caltech, n_estimators=i, n_jobs=-1, random_state=42)


=== Random Forest Baseline: Caltech101  ===
>>> [Caltech101] Training RandomForest with 115 estimators:
Training Random Forest with n_estimators=115...
Evaluating Random Forest...
Random Forest Results -> Accuracy: 0.4758 | Precision: 0.4272 | Recall: 0.4758 | F1: 0.4080

Classification Report (Random Forest):
              precision    recall  f1-score   support

           0       0.41      0.89      0.56        44
           1       0.69      0.95      0.80        44
           2       0.62      1.00      0.77        20
           3       0.70      0.93      0.80        80
           4       0.38      0.60      0.46         5
           5       0.51      0.97      0.67        80
           6       0.00      0.00      0.00         4
           7       0.00      0.00      0.00         4
           8       0.00      0.00      0.00         5
           9       0.00      0.00      0.00         5
          10       0.25      0.20      0.22         5
          11       1.00      0.67     